In [1]:
!pip install pyspark
!pip install -U -q PyDrive
!apt install openjdk-8-jdk-headless -qq

The following additional packages will be installed:
  libxtst6 openjdk-8-jre-headless
Suggested packages:
  openjdk-8-demo openjdk-8-source libnss-mdns fonts-dejavu-extra fonts-nanum
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  libxtst6 openjdk-8-jdk-headless openjdk-8-jre-headless
0 upgraded, 3 newly installed, 0 to remove and 35 not upgraded.
Need to get 39.7 MB of archives.
After this operation, 144 MB of additional disk space will be used.
Selecting previously unselected package libxtst6:amd64.
(Reading database ... 126308 files and directories currently installed.)
Preparing to unpack .../libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package openjdk-8-jre-headless:amd64.
Preparing to unpack .../openjdk-8-jre-headless_8u452-ga~us1-0ubuntu1~22.04_amd64.deb ...
Unpacking openjdk-8-jre-headless:amd64 (8u452-ga~us1-0

In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

In [4]:
from pyspark.context import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, to_date, when, regexp_replace
from pyspark.sql.types import IntegerType, DateType

In [5]:
sc = SparkContext.getOrCreate()
spark = SparkSession.builder.appName(
'DrivenData Distributed Computing').getOrCreate()

In [6]:
df_14 = spark.read.csv('batch_2025-04-02.csv', header=True,
inferSchema=True)
df_15 = spark.read.csv('batch_2025-04-03.csv', header=True,
inferSchema=True)


In [7]:
df = df_14.union(df_15)

In [8]:
df.printSchema()
df.show(5)
df.describe().show()


root
 |-- person_name: string (nullable = true)
 |-- user_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- personal_number: long (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- address: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- mac_address: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- iban: string (nullable = true)
 |-- accessed_at: timestamp (nullable = true)
 |-- session_duration: integer (nullable = true)
 |-- download_speed: integer (nullable = true)
 |-- upload_speed: integer (nullable = true)
 |-- consumed_traffic: integer (nullable = true)
 |-- unique_id: string (nullable = true)

+--------------+-------------+--------------------+---------------+----------+--------------------+------------+-----------------+--------------+--------------------+-------------------+----------------+--------------+------------+----------------+--------------------+
|   person_name|    user_name|              

In [10]:
df = df.na.fill({"email": "unknown@example.com", "phone": "000-000-0000"})


In [11]:
df = df.dropDuplicates(subset=["unique_id"])


In [12]:
df = df.withColumn("birth_date", col("birth_date").cast(DateType()))
df = df.withColumn( "session_duration", col("session_duration").cast(IntegerType()))


In [13]:
df.printSchema()

root
 |-- person_name: string (nullable = true)
 |-- user_name: string (nullable = true)
 |-- email: string (nullable = false)
 |-- personal_number: long (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- address: string (nullable = true)
 |-- phone: string (nullable = false)
 |-- mac_address: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- iban: string (nullable = true)
 |-- accessed_at: timestamp (nullable = true)
 |-- session_duration: integer (nullable = true)
 |-- download_speed: integer (nullable = true)
 |-- upload_speed: integer (nullable = true)
 |-- consumed_traffic: integer (nullable = true)
 |-- unique_id: string (nullable = true)



In [22]:
df_filtered = df.filter(to_date(df.accessed_at) > '2024-10-13')
df_filtered = df.filter(df.consumed_traffic> 1000)


In [25]:
df_grouped = df.groupBy("person_name").agg( {"session_duration": "avg", "consumed_traffic": "sum"})
df_grouped.show()


+------------------+---------------------+---------------------+
|       person_name|sum(consumed_traffic)|avg(session_duration)|
+------------------+---------------------+---------------------+
| Leopoldina Manole|              7520978|              10683.0|
|     Malvina Aanei|              2372335|              14407.0|
|  Olimpian Cristea|              1945924|              13733.6|
|        Nicuță Ene|              2036012|              19169.5|
|Gregorian Dochioiu|              4403697|               9551.8|
|Augustin Gheorghiu|             10766823|              19086.5|
|     Iosefina Niță|              7067156|   19953.166666666668|
|         Stela Ene|              7020911|   11647.333333333334|
|      Alberta Popa|              3424330|   21906.333333333332|
|   Ortansa Cristea|              3421945|              13223.0|
|       Raul Stancu|              8562636|   15592.714285714286|
|       Emil Nistor|              7256102|            16029.125|
| Mariana Georgescu|     

In [30]:
df = df.withColumn( "total_bandwidth", col("download_speed") + col("upload_speed"))
df = df.withColumn("birth_year", expr("year(birth_date)"))
df = df.withColumn("activity_level", when(col("session_duration") > 120, "active")
                    .when(col("session_duration") .between(30, 120),"moderate")
                    .otherwise("less_active"))
df = df.withColumn("masked_email", regexp_replace("email", "(\\w{3})\\w+@(\\w+)","$1***@$2"))
df.show()

+----------------+---------------+--------------------+---------------+----------+--------------------+------------+-----------------+---------------+--------------------+-------------------+----------------+--------------+------------+----------------+--------------------+---------------+----------+--------------+-------------------+
|     person_name|      user_name|               email|personal_number|birth_date|             address|       phone|      mac_address|     ip_address|                iban|        accessed_at|session_duration|download_speed|upload_speed|consumed_traffic|           unique_id|total_bandwidth|birth_year|activity_level|       masked_email|
+----------------+---------------+--------------------+---------------+----------+--------------------+------------+-----------------+---------------+--------------------+-------------------+----------------+--------------+------------+----------------+--------------------+---------------+----------+--------------+----------

In [31]:
df.agg({"session_duration": "avg"}).show()
df.agg({"session_duration": "max"}).show()


+---------------------+
|avg(session_duration)|
+---------------------+
|   18013.524390727558|
+---------------------+

+---------------------+
|max(session_duration)|
+---------------------+
|                36000|
+---------------------+



In [32]:
df_ip_activity = df.groupBy("ip_address").agg(
 {"consumed_traffic": "sum"}).orderBy(
 "sum(consumed_traffic)", ascending=False)
df_ip_activity.show()

+---------------+---------------------+
|     ip_address|sum(consumed_traffic)|
+---------------+---------------------+
|  83.254.182.79|              1999994|
|   84.21.98.211|              1999994|
|   14.89.68.181|              1999989|
|     54.5.2.108|              1999970|
|194.167.244.174|              1999957|
| 137.39.230.122|              1999944|
|  49.40.116.249|              1999923|
|109.182.183.122|              1999904|
|   1.73.217.216|              1999881|
|182.170.112.252|              1999855|
| 113.21.172.130|              1999826|
| 81.186.228.202|              1999810|
| 110.10.211.161|              1999794|
| 17.129.252.148|              1999777|
|   6.159.183.98|              1999775|
|    5.6.136.179|              1999769|
| 148.155.52.104|              1999760|
|    83.63.42.98|              1999757|
|   14.69.90.249|              1999729|
| 156.174.46.239|              1999719|
+---------------+---------------------+
only showing top 20 rows



In [33]:
df_time = df.withColumn("access_date",
to_date("accessed_at")).groupBy("access_date").count()
df_time.show()

+-----------+-----+
|access_date|count|
+-----------+-----+
| 2024-09-18|  289|
| 2025-02-16|  294|
| 2024-05-30|  293|
| 2024-06-04|  257|
| 2024-06-12|  279|
| 2024-08-27|  271|
| 2025-02-15|  294|
| 2024-11-25|  265|
| 2024-05-25|  270|
| 2024-10-24|  287|
| 2024-11-02|  288|
| 2025-02-01|  268|
| 2025-03-23|  260|
| 2024-04-20|  289|
| 2025-02-05|  303|
| 2024-11-23|  262|
| 2024-08-30|  303|
| 2024-10-02|  288|
| 2025-02-13|  280|
| 2024-07-08|  269|
+-----------+-----+
only showing top 20 rows



In [34]:
df.write.csv("processed_data_2024-10-15.csv", header=True)
